In [1]:
import tensorflow as tf
from transformers import BertTokenizer, TFBertModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import pandas as pd
import numpy as np


/Users/yuvrajbhatkariya/data/VScode.C++/Python/myenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv("dataset/spam.csv")  # adjust filename
df = df[['Message', 'Category']]    # ensure columns are correct
df['Category'] = df['Category'].map({'ham': 0, 'spam': 1})  # if needed


In [3]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Tokenize and encode the texts
X = list(df['Message'])
y = list(df['Category'])

encodings = tokenizer(X, truncation=True, padding=True, max_length=128, return_tensors='tf')


In [4]:
import numpy as np

input_ids = np.array(encodings['input_ids'])
attention_masks = np.array(encodings['attention_mask'])

train_inputs, test_inputs, train_labels, test_labels = train_test_split(
    input_ids, y, test_size=0.2, random_state=42
)

train_attention_masks, test_attention_masks = train_test_split(
    attention_masks, test_size=0.2, random_state=42
)


In [5]:
def create_model():
    input_ids = tf.keras.Input(shape=(128,), dtype=tf.int32, name='input_ids')
    attention_mask = tf.keras.Input(shape=(128,), dtype=tf.int32, name='attention_mask')

    bert_model = TFBertModel.from_pretrained('bert-base-uncased')
    bert_output = bert_model(input_ids, attention_mask=attention_mask)[1]  # pooled output

    output = tf.keras.layers.Dense(1, activation='sigmoid')(bert_output)

    model = tf.keras.Model(inputs=[input_ids, attention_mask], outputs=output)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=2e-5),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model


In [6]:
from transformers import TFBertForSequenceClassification

model = TFBertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

All PyTorch model weights were used when initializing TFBertForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [7]:
from tensorflow.keras.optimizers import Adam
model.compile(
    optimizer='adam', # Use the string identifier for the optimizer
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

history = model.fit(
    {'input_ids': train_inputs, 'attention_mask': train_attention_masks},
    np.array(train_labels), # Convert train_labels to a numpy array
    validation_data=(
        {'input_ids': test_inputs, 'attention_mask': test_attention_masks},
        np.array(test_labels) # Convert test_labels to a numpy array
    ),
    epochs=5,
    batch_size=32
)

Epoch 1/5
73/73 [==============================] - 1102s 15s/step - loss: 0.7401 - accuracy: 0.4801 - val_loss: 0.6974 - val_accuracy: 0.4922
Epoch 2/5
73/73 [==============================] - 871s 12s/step - loss: 0.7020 - accuracy: 0.4827 - val_loss: 0.7039 - val_accuracy: 0.4922
Epoch 3/5
73/73 [==============================] - 864s 12s/step - loss: 0.7122 - accuracy: 0.5082 - val_loss: 0.7109 - val_accuracy: 0.5078
Epoch 4/5
73/73 [==============================] - 1868s 26s/step - loss: 0.7071 - accuracy: 0.4866 - val_loss: 0.7233 - val_accuracy: 0.4922
Epoch 5/5
73/73 [==============================] - 1421s 20s/step - loss: 0.7060 - accuracy: 0.4896 - val_loss: 0.6931 - val_accuracy: 0.5078


In [9]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
tokenizer.save_pretrained("saved_model_directory")

('saved_model_directory/tokenizer_config.json',
 'saved_model_directory/special_tokens_map.json',
 'saved_model_directory/vocab.txt',
 'saved_model_directory/added_tokens.json')